# Model Evaluation

In this lesson, we'll learn how to evaluate the quality of a machine learning model. By the end of this lesson, students will be able to:

- Apply `get_dummies` to represent categorical features as one or more dummy variables.
- Apply `train_test_split` to randomly split a dataset into a training set and test set.
- Evaluate machine learning models in terms of overfit and underfit.

In [1]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error

import pandas as pd

## Dummy variables

The `homes` dataset has 492 homes in New York (`NY`) and San Francisco (`SF`). Each row records a home's beds, baths, price, year built, square footage, price per square foot, elevation, and city. Let's try to predict the **price** of a home from all the other columns.

In [2]:
homes = pd.read_csv("homes.csv")
homes

,beds,bath,price,year_built,sqft,price_per_sqft,elevation,city
0,2.0,1.0,999000,1960,1000,999,10,NY
1,2.0,2.0,2750000,2006,1418,1939,0,NY
2,2.0,2.0,1350000,1900,2150,628,9,NY
3,1.0,1.0,629000,1903,500,1258,9,NY
4,0.0,1.0,439000,1930,500,878,10,NY
...,...,...,...,...,...,...,...,...
487,5.0,2.5,1800000,1890,3073,586,76,SF
488,2.0,1.0,695000,1923,1045,665,106,SF
489,3.0,2.0,1650000,1922,1483,1113,106,SF
490,1.0,1.0,649000,1983,850,764,163,SF


There's a problem: the `city` column contains *categorical* string values, and `LinearRegression` can't multiply a string like `"NY"` by a *numeric* slope value. Let's learn how to represent these categorical features as one or more dummy variables.

A **dummy variable** is a 0/1 column for one category. [`pd.get_dummies`](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html) replaces each string column with one dummy column per category: `city` becomes `city_NY` (1 if the home is in NY, else 0) and `city_SF`. Numeric columns are left alone.

**Try it:** run the cell below as is. It fails with `ValueError: could not convert string to float: 'NY'`. Fix it by wrapping `homes.drop("price", axis=1)` in `pd.get_dummies(...)`.

In [3]:
def model_parameters(reg, columns):
    """Returns a string with the linear regression model parameters for the given column names."""
    slopes = [f"{coef:.2f}({columns[i]})" for i, coef in enumerate(reg.coef_)]
    return " + ".join([f"{reg.intercept_:.2f}"] + slopes)

In [4]:
# Convert the categorical city column into 0/1 dummy columns.
X = pd.get_dummies(homes.drop("price", axis=1))
y = homes["price"]
reg = LinearRegression().fit(X, y)

print("Model:", model_parameters(reg, X.columns))
print("Error:", mean_squared_error(y, reg.predict(X)))

Model: 2667451.86 + -91657.73(beds) + -46672.61(bath) + -2986.99(year_built) + 1611.73(sqft) + 2563.08(price_per_sqft) + -1081.63(elevation) + -173696.89(city_NY) + 173696.89(city_SF)
Error: 976495129525.2129


Once it runs, look at the features the model actually used. Where did the `city` column go?

In [ ]:
X.head()

A few things to notice about this model:

- The error is huge (about 976,000,000,000) because MSE is in **squared dollars**. Below, we'll switch to **RMSE** (the square root of MSE), which is back in dollars.
- The `city_NY` and `city_SF` slopes are exactly opposite (about −173,697 and +173,697).

**Question:** Why are the two city slopes exactly opposite? *Hint:* for any home, what is `city_NY + city_SF`?

Because every row belongs to exactly one city, `city_NY + city_SF` is always 1. The two columns contain the same information, so the model can shift the same city effect between the two slopes and the intercept. This is why the slopes have equal size and opposite signs.

**Dropping the redundant column.** Since `city_NY + city_SF` is always 1, one of the two columns is redundant: if `city_SF` is 0, the home must be in NY. Passing `drop_first=True` to `pd.get_dummies` drops the first category (`city_NY`). NY becomes the **baseline**: the intercept describes a NY home, and the `city_SF` slope is how much *more* an SF home costs than an otherwise identical NY home.

Run the cell below and compare the model to the one above.

In [ ]:
X = pd.get_dummies(homes.drop("price", axis=1), drop_first=True)
reg = LinearRegression().fit(X, y)

print("Model:", model_parameters(reg, X.columns))
print("Error:", mean_squared_error(y, reg.predict(X)))

**Question:** The error is exactly the same as before. Why didn't dropping a column make the model worse? How does the `city_SF` slope compare to the two city slopes in the first model?

Dropping one dummy column does not remove useful information: the remaining column tells us whether a home is in SF, and a 0 means it is in NY. The model makes the same predictions, so its error stays the same. The `city_SF` slope in the second model is about twice the positive `city_SF` slope in the first model because the second model compares SF directly with NY as the baseline.

## Overfitting

In our introduction to machine learning, we explained that the researchers who worked on calibrating the PurpleAir Sensor (PAS) measurements against the EPA Air Quality Sensor (AQS) measurements ultimately decided to use the model that included only the PAS and humidity features (variables)—ignoring the opportunity to use the temperature and dew point even though a model that includes all features produced a lower overall mean squared error.

In [5]:
sensor_data = pd.read_csv("sensor_data.csv")
sensor_data

,AQS,temp,humidity,dew,PAS
0,6.7,18.027263,38.564815,3.629662,8.616954
1,3.8,16.115280,49.404315,5.442318,3.493916
2,4.0,19.897634,29.972222,1.734051,3.799601
3,4.7,21.378334,32.474513,4.165624,4.369691
4,3.2,18.443822,43.898226,5.867611,3.191071
...,...,...,...,...,...
12092,5.5,-12.101337,54.188889,-19.555834,2.386120
12093,16.8,4.159967,56.256030,-3.870659,32.444987
12094,15.6,1.707895,65.779221,-4.083768,25.297018
12095,14.0,-14.380144,48.206481,-23.015378,8.213208


In [ ]:
X = sensor_data.drop("AQS", axis=1)
y = sensor_data["AQS"]
reg = LinearRegression().fit(X, y)

print("Model:", model_parameters(reg, X.columns))
# RMSE, or root mean squared error, is the square root of the mean of the squared errors.
print("Error:", root_mean_squared_error(y, reg.predict(X)))

In fact, the authors ([Barkjohn et al. 2021](https://doi.org/10.5194/amt-14-4617-2021
)) tested the impact of incrementally introducing a feature to the model to determine which combination of features could provide the most useful model—not necessarily the most accurate one. In their research, they aimed to predict the PAS measurement from the AQS measurement, the opposite of our task, and they included interactions between features indicated by the × symbol. For each grouping of models with a certain number of features, they highlighted the model with the lowest **root mean squared error** (RMSE, or the square root of the MSE) using an asterisk `*`.

![Table 2 from Barkjohn et al. 2021 depicting every combination of features used in linear models](https://amt.copernicus.org/articles/14/4617/2021/amt-14-4617-2021-t02.png)

The model at the bottom of the table that includes all the features also has the lowest RMSE loss. But the loss value alone is not a convincing measure: adding more features into a model not only leads to a model that is harder to explain, but also increases the possibility of overfitting.

A model is considered **overfit** if its predictions correspond too closely to the *training dataset* such that improvements reported on the training dataset are not reflected when the model is run on a *testing dataset* (or in the real world). To simulate training and testing datasets, we can take our `X` features and `y` labels and subdivide them into `X_train, X_test, y_train, y_test` using the `train_test_split` function. **The testing dataset should only be used during final model evaluation when we're ready to report the overall effectiveness of our final machine learning model.**

**Connecting to Curve of Best Fit:** in [Curve of Best Fit](../classwork/cw-best-fit.ipynb), the degree 10 curve had the lowest error on the *training* points but terrible error on the *future* points. It memorized the noise instead of learning the trend. That's exactly this definition of overfitting. The pieces line up:

| Curve of Best Fit | Here |
|---|---|
| polynomial degree (more coefficients) | number of features (more coefficients) |
| `train` points | training dataset |
| `future` points | testing dataset |
| the `split` column we gave you | `train_test_split`, which splits for you at random |

In [6]:
from sklearn.model_selection import train_test_split

# Rebuild X and y from the sensor data, so this cell is safe to re-run
# even after later cells have changed X and y
X = sensor_data.drop("AQS", axis=1)
y = sensor_data["AQS"]

# test_size=0.2 indicates 80% training dataset and 20% testing dataset
# random_state=42 makes the "random" split the same every run, so we all get the same numbers
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# During model training, use only the training dataset
reg = LinearRegression().fit(X_train, y_train)

print("Model:", model_parameters(reg, X_train.columns))
print("Training error:", root_mean_squared_error(y_train, reg.predict(X_train)))
# During model evaluation, use the testing dataset
print("Testing error: ", root_mean_squared_error(y_test, reg.predict(X_test)))

Model: 6.96 + -0.02(temp) + -0.08(humidity) + 0.03(dew) + 0.43(PAS)
Training error: 2.231554848636717
Testing error:  2.407392355993048


**Question:** Is this model overfit? Compare the training error to the testing error. What would you expect to see if it *were* overfit?

*Hint:* in Curve of Best Fit, degree 10 had 11 coefficients to fit just 12 points, so it could pass through all of them. How many coefficients does this model have, and how many training rows?

This model is not strongly overfit: the training and testing RMSE values should be reasonably close. The model has one coefficient for each feature plus an intercept, so it has 5 coefficients total. The training split has 80% of 492 rows, or about 393 rows, which is far more observations than coefficients. If the model were overfit, the training error would be much smaller than the testing error.

## Feature selection

**Feature selection** describes the process of selecting only a subset of features in order to improve the quality of a machine learning model. In the air quality sensor calibration study, we can begin with all the features and gradually remove the least-important features one-by-one.

**Try it:** the cell below starts with all four features. Remove one feature at a time, re-run, and note the error each time.

In [14]:
# Start with all four sensor features.
all_features = ["PAS", "humidity", "temp", "dew"]

# Measure the model with every feature.
full_model = LinearRegression().fit(X_train[all_features], y_train)
full_error = root_mean_squared_error(y_test, full_model.predict(X_test[all_features]))
print("All features:", full_error)

# Remove one feature at a time and compare the testing RMSE.
for feature_to_remove in all_features:
    selected_features = [feature for feature in all_features if feature != feature_to_remove]
    model = LinearRegression().fit(X_train[selected_features], y_train)
    error = root_mean_squared_error(y_test, model.predict(X_test[selected_features]))
    print("Without", feature_to_remove + ":", error)

All features: 2.407392355993046
Without PAS: 5.543159453802769
Without humidity: 2.446355404558928
Without temp: 2.4089946846810055
Without dew: 2.4082756790117514


**Question:** Which features could you remove with almost no change in error? Which feature hurt the most when removed?

Removing `temp` or `dew` changes the testing RMSE only slightly, so either feature could be removed with little effect. Removing `humidity` increases the error a little more. Removing `PAS` hurts the most: the RMSE increases from about 2.41 to 5.54, so `PAS` is the most important feature here.

[Recursive feature elimination](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html) (RFE) automates this process by starting with a model that includes all the variables and removes features from the model starting with the ones that contribute the least weight (smallest coefficients in a linear regression).

In [10]:
from sklearn.feature_selection import RFE

# Remove 1 feature per step until half the original features remain
rfe = RFE(LinearRegression(), step=1, n_features_to_select=0.5, verbose=1)
rfe.fit(X_train, y_train)

# Show the final subset of features
rfe_features = X.columns[[r == 1 for r in rfe.ranking_]]
print("Features:", list(rfe_features))

# Extract the last LinearRegression model trained on the final subset of features
reg = rfe.estimator_

print("Model:", model_parameters(reg, rfe_features))
print("Error:", root_mean_squared_error(y_test, rfe.predict(X_test)))

Fitting estimator with 4 features.
Fitting estimator with 3 features.
Features: ['humidity', 'PAS']
Model: 6.23 + -0.07(humidity) + 0.43(PAS)
Error: 2.407415966891961


## Cross validation

How do we know when to stop removing features during feature selection? We can certainly use intuition or look at the changes in error as we remove each feature. But this still requires us to evaluate the model somehow. If the testing dataset can only be used at the end of model evaluation, **it was wrong of us to use the testing dataset during feature selection**!

The same goes for Curve of Best Fit: you chose the degree by checking the error on the future points. That's using the test set to make a choice, so the future error you reported was no longer a fair measure.

Cross-validation provides one way to help us explore different models before we choose a final model for evaluation at the end. [**Cross-validation**](https://scikit-learn.org/stable/modules/cross_validation.html) lets us evaluate models without touching the testing dataset by introducing new *validation datasets*.

To show cross-validation, we'll use a different kind of model: a **decision tree** (`DecisionTreeRegressor`). It predicts by asking a series of yes/no questions about the features (such as "Is PAS < 20?"), and its `max_depth` setting limits how many questions in a row it can ask. A setting like this, which we choose *before* training, is called a **hyperparameter**. We'll look inside decision trees in the [Decision Trees](decision-trees.ipynb) lesson; for now, treat `max_depth` as a knob to tune.

The simplest way to cross-validate is to call the `cross_val_score` helper function on an unfitted machine learning algorithm and the dataset. This function will further subdivide the training dataset into 5 folds and, for each of the 5 folds, train a separate model on the training folds and evaluate them on the validation fold.

![A depiction of a 5 fold cross validation on a training set, while holding out a test set](https://scikit-learn.org/stable/_images/grid_search_cross_validation.png)

The `scoring` parameter can accept the string name of [a scorer function](https://scikit-learn.org/stable/modules/model_evaluation.html#scoring-parameter). Higher (more positive) scores are considered better, so we use the negative RMSE value as the scorer function.

In [11]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor

cross_val_score(
    estimator=DecisionTreeRegressor(max_depth=2),
    X=X_train,
    y=y_train,
    scoring="neg_root_mean_squared_error",
    verbose=3,
)

[CV] END ............................... score: (test=-2.804) total time=   0.0s
[CV] END ............................... score: (test=-2.989) total time=   0.0s
[CV] END ............................... score: (test=-3.198) total time=   0.0s
[CV] END ............................... score: (test=-2.874) total time=   0.0s
[CV] END ............................... score: (test=-2.874) total time=   0.0s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.1s finished


array([-2.80365003, -2.98865474, -3.19838177, -2.87416817, -2.87397044])

As we've seen throughout these lessons on machine learning, we prefer to automate our processes. Rather than modify the `max_depth` and manually tweaking the values until we find something that works, we can use `GridSearchCV` to exhaustively search all hyperparameter options. Here, the first 5 folds for `max_depth=2` are exactly the same as `cross_val_score`.

`max_depth` plays the same role for a tree that the degree played for a polynomial: higher means more flexible and more likely to overfit. `GridSearchCV` does what your degree loop did in Curve of Best Fit, but it scores each option on *validation* folds carved out of the training set, so the test set stays untouched.

In [12]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(
    estimator=DecisionTreeRegressor(),
    param_grid={"max_depth": [2, 3, 4, 5, 6, 7, 8, 9, 10]},
    scoring="neg_root_mean_squared_error",
    verbose=3,
)
search.fit(X_train, y_train)

# Show the best score and best estimator at the end of hyperparameter search
print("Mean score for best model:", search.best_score_)
reg = search.best_estimator_
print("Best model:", reg)

Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV 1/5] END ......................max_depth=2;, score=-2.804 total time=   0.0s
[CV 2/5] END ......................max_depth=2;, score=-2.989 total time=   0.0s
[CV 3/5] END ......................max_depth=2;, score=-3.198 total time=   0.0s
[CV 4/5] END ......................max_depth=2;, score=-2.874 total time=   0.0s
[CV 5/5] END ......................max_depth=2;, score=-2.874 total time=   0.0s
[CV 1/5] END ......................max_depth=3;, score=-2.599 total time=   0.0s
[CV 2/5] END ......................max_depth=3;, score=-2.733 total time=   0.0s
[CV 3/5] END ......................max_depth=3;, score=-2.875 total time=   0.0s
[CV 4/5] END ......................max_depth=3;, score=-2.630 total time=   0.0s
[CV 5/5] END ......................max_depth=3;, score=-2.602 total time=   0.0s
[CV 1/5] END ......................max_depth=4;, score=-2.328 total time=   0.0s
[CV 2/5] END ......................max_depth=4;, 

Finally, we can report the test error for the best model by evaluating it against the testing dataset. **Question:** Why is the testing dataset error different from the mean score for the best model printed above?

In [13]:
print("Error:", root_mean_squared_error(y_test, reg.predict(X_test)))

Error: 2.541820041201662


The cross-validation score is the average validation RMSE across several folds of the training data. The testing error is calculated once on a separate set of rows that was never used during the search. These are different rows and different calculations, so their values do not need to match.

## Checkoff

Put it all together on the `homes` dataset. Show your teacher your output and answers.

**Use new variable names** (`X_homes`, `y_homes`, `X_homes_train`, ...), not `X`, `y` and `X_train`. Otherwise you overwrite the sensor data variables, and re-running cells above will quietly use the homes data instead.

1. Build `X_homes` with `pd.get_dummies` (as in the first section) and `y_homes` as the price.
2. Split with `train_test_split(X_homes, y_homes, test_size=0.2, random_state=42)` into `X_homes_train, X_homes_test, y_homes_train, y_homes_test`.
3. Fit a `LinearRegression` on the **training** set only.
4. Print the training RMSE and the testing RMSE.

In [8]:
# Build features and target. Dummy variables represent the city as numbers.
X_homes = pd.get_dummies(homes.drop("price", axis=1))
y_homes = homes["price"]

# Keep the test data separate until the final evaluation.
X_homes_train, X_homes_test, y_homes_train, y_homes_test = train_test_split(
    X_homes, y_homes, test_size=0.2, random_state=42
)

# Train only on the training data.
homes_model = LinearRegression().fit(X_homes_train, y_homes_train)

print("Training RMSE:", root_mean_squared_error(
    y_homes_train, homes_model.predict(X_homes_train)
))
print("Testing RMSE:", root_mean_squared_error(
    y_homes_test, homes_model.predict(X_homes_test)
))

Training RMSE: 1021718.862845096
Testing RMSE: 899040.755195818


Now repeat the same steps, but also drop the `price_per_sqft` column from `X_homes` (e.g. `X_homes.drop("price_per_sqft", axis=1)`). What happens to the testing RMSE?

In [9]:
# Remove price_per_sqft because it is calculated from price, the target.
X_homes_without_price_per_sqft = X_homes.drop("price_per_sqft", axis=1)

# Use the same split so this is a fair comparison.
X_homes_train_without_price_per_sqft, X_homes_test_without_price_per_sqft, y_homes_train, y_homes_test = train_test_split(
    X_homes_without_price_per_sqft,
    y_homes,
    test_size=0.2,
    random_state=42,
)

homes_model_without_price_per_sqft = LinearRegression().fit(
    X_homes_train_without_price_per_sqft, y_homes_train
)

print("Training RMSE without price_per_sqft:", root_mean_squared_error(
    y_homes_train,
    homes_model_without_price_per_sqft.predict(X_homes_train_without_price_per_sqft),
))
print("Testing RMSE without price_per_sqft:", root_mean_squared_error(
    y_homes_test,
    homes_model_without_price_per_sqft.predict(X_homes_test_without_price_per_sqft),
))

Training RMSE without price_per_sqft: 1665171.2856898918
Testing RMSE without price_per_sqft: 1602586.2628207945


**Question:** `price_per_sqft` made the model much more accurate. Why is it unfair (cheating) to use it as a feature to predict `price`? *Hint:* how would you calculate `price_per_sqft` for a home?

`price_per_sqft` is calculated from the target price: `price_per_sqft = price / sqft`. Therefore, using it to predict `price` gives the model information derived from the answer it is supposed to predict. This is data leakage, so the very low error would not represent a fair prediction of a new home's price.

**Question:** When we tuned `max_depth` with `GridSearchCV`, why did we only give it `X_train` and `y_train`, never `X_test` and `y_test`?

We gave `GridSearchCV` only the training data so it could use validation folds to choose `max_depth` without seeing the final test examples. If we used the test data while tuning, the model choice would be influenced by those examples, and the final testing error would no longer be an unbiased estimate of performance on new data.